# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [ ]:

from sklearn.preprocessing import StandardScaler

# Drop missing values
spaceship = spaceship.dropna()

# Extract deck letter from Cabin
spaceship["Cabin"] = spaceship["Cabin"].str[0]

# Drop non-informative columns
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

# One-hot encode categorical and boolean columns
categorical_cols = spaceship.select_dtypes(include=["object", "bool"]).columns
spaceship = pd.get_dummies(spaceship, columns=categorical_cols, drop_first=True)

# Split features and target
X = spaceship.drop("Transported_True", axis=1)
y = spaceship["Transported_True"]

# Scale numerical features
numerical_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:

from sklearn.ensemble import GradientBoostingClassifier

# Best model from previous lab: Gradient Boosting with default hyperparameters
gb_default = GradientBoostingClassifier(random_state=42)
gb_default.fit(X_train, y_train)


- Evaluate your model

In [ ]:

from sklearn.metrics import accuracy_score, classification_report

y_pred_default = gb_default.predict(X_test)
acc_default = accuracy_score(y_test, y_pred_default)

print(f"Default Gradient Boosting Accuracy: {acc_default:.4f}")
print()
print(classification_report(y_test, y_pred_default))


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [ ]:

param_grid = {
    "n_estimators":   [100, 200, 300],
    "learning_rate":  [0.05, 0.1, 0.2],
    "max_depth":      [3, 4, 5],
    "min_samples_split": [2, 5],
}

print("Total combinations:", 
      len(param_grid["n_estimators"]) *
      len(param_grid["learning_rate"]) *
      len(param_grid["max_depth"]) *
      len(param_grid["min_samples_split"]))


- Run Grid Search

In [ ]:

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")


- Evaluate your model

In [ ]:

best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"Default model accuracy: {acc_default:.4f}")
print(f"Tuned model accuracy:   {acc_tuned:.4f}")
print(f"Improvement:            {acc_tuned - acc_default:+.4f}")
print()
print(classification_report(y_test, y_pred_tuned))
